In [ ]:
from collections import defaultdict
import pickle
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
import torch_geometric.transforms as T
from torch_geometric.loader import LinkNeighborLoader, NeighborLoader

from tqdm import tqdm


device = 'cuda' if torch.cuda.is_available() else 'cpu'



##  Следующие две ячейки по факту дублируют MultiMult.ipynb, т.к. в этом файле используются результаты MDM

In [ ]:
#класс датасета для модели
class GraphDataset(Dataset):
    def __init__(self, h_local, h_type, rel, t_local, t_type):
        self.h_local = h_local
        self.h_type = h_type
        self.rel = rel
        self.t_local = t_local
        self.t_type = t_type

    def __len__(self):
        return len(self.h_local)

    def __getitem__(self, idx):
        return (
            self.h_local[idx],
            self.h_type[idx],
            self.rel[idx],
            self.t_local[idx],
            self.t_type[idx]
        )

#класс модели
class MultiModalDistMult(nn.Module):
    def __init__(self, raw_tensors, output_dim, num_relations):
        super().__init__()
        self.output_dim = output_dim
        self.num_types = len(raw_tensors)

        #хранилище для сырых эмбеддингов
        self.raw_stores = nn.ModuleDict({
            str(t_id): nn.Embedding.from_pretrained(tensor, freeze=True)
            for t_id, tensor in raw_tensors.items()
        })

        #создаем MLP-проекторы автоматически под размер входа каждого тензора
        self.projections = nn.ModuleDict()
        for t_id, tensor in raw_tensors.items():
            in_dim = tensor.shape[1] #узнаем исходную размерность одного типа
            self.projections[str(t_id)] = nn.Sequential( #создаем проектор
                nn.Linear(in_dim, output_dim * 2),
                nn.ReLU(),
                nn.LayerNorm(output_dim * 2),
                nn.Linear(output_dim * 2, output_dim)
            )

        #эмбеддинги отношений
        self.rel_emb = nn.Embedding(num_relations, output_dim)

    def forward(self, h_local, h_type, r_idx, t_local, t_type):
        batch_size = h_local.size(0)
        device = h_local.device

        #заготовки для итоговых тензоров
        h_proj = torch.zeros(batch_size, self.output_dim, device=device)
        t_proj = torch.zeros(batch_size, self.output_dim, device=device)

        #проход по всем типам данных
        for type_id in range(self.num_types):
            t_str = str(type_id)

            #выбирам элементы текущего типа
            h_mask = (h_type == type_id)
            t_mask = (t_type == type_id)

            #проекция голов
            if h_mask.any():
                raw_h = self.raw_stores[t_str](h_local[h_mask]).float()
                h_proj[h_mask] = self.projections[t_str](raw_h)

            #проекция хвостов
            if t_mask.any():
                raw_t = self.raw_stores[t_str](t_local[t_mask]).float()
                t_proj[t_mask] = self.projections[t_str](raw_t)

        #L2 нормализация
        h_proj = F.normalize(h_proj, p=2, dim=1)
        t_proj = F.normalize(t_proj, p=2, dim=1)

        #берем нужные эмбеддинги отношений
        r = self.rel_emb(r_idx)

        #считаем скор
        score = torch.sum(h_proj * r * t_proj, dim=1)
        return score


#функция для подготовки к оценке
def prepare_evaluation_mdm(model, all_h_local, all_h_type, all_r, all_t_local, all_t_type, device):
    model.eval()

    #проецируем все узлы графа в общее пространство
    all_embs = []
    entity2eval_id = {} #(local_id, type_id) -> индекс в матрице оценки
    eval_id = 0

    with torch.no_grad():
        #проход по всем типам данных
        for type_id in range(model.num_types):
            type_str = str(type_id)
            num_nodes = model.raw_stores[type_str].weight.size(0)

            #проецируем все узлы этого типа
            locs = torch.arange(num_nodes, device=device)
            raw = model.raw_stores[type_str](locs).float()
            proj = model.projections[type_str](raw)
            proj = F.normalize(proj, p=2, dim=1)

            all_embs.append(proj)

            #заполняем словарь какая строка новой матрицы какому узлу принадлежит
            for local_id in range(num_nodes):
                entity2eval_id[(local_id, type_id)] = eval_id
                eval_id += 1

    #матрица всех узлов графа [N_total, D]
    E_all = torch.cat(all_embs, dim=0)

    #ссловарь для фильтрации (h_eval_id, r) -> set(t_eval_ids)
    filter_dict = defaultdict(set)

    #проходимся по всем ребрам графа чтобы собрать словарь
    for i in range(len(all_h_local)):
        h_key = (all_h_local[i].item(), all_h_type[i].item())
        t_key = (all_t_local[i].item(), all_t_type[i].item())
        r = all_r[i].item()

        h_eval_id = entity2eval_id[h_key]
        t_eval_id = entity2eval_id[t_key]

        filter_dict[(h_eval_id, r)].add(t_eval_id)

    return E_all, entity2eval_id, filter_dict


#функция оценки
def evaluate_filtered_mdm(model, val_dataloader, E_all, entity_to_eval_id, filter_dict, device, k_list=[1, 5, 10, 50, 100]):
    model.eval()

    mrr = 0
    hits = {k: 0 for k in k_list}
    total_samples = 0

    with torch.no_grad():
        for i, batch in tqdm(enumerate(val_dataloader)):

            h_loc, h_typ, r_idx, t_loc, t_typ = [b.to(device) for b in batch]
            batch_size = h_loc.size(0)
            total_samples += batch_size

            #берем векторы голов и отношений
            h_eval_ids = [entity_to_eval_id[(local.item(), typ.item())] for local, typ in zip(h_loc, h_typ)]
            h_proj = E_all[torch.tensor(h_eval_ids, device=device)]
            r_emb = model.rel_emb(r_idx)

            #считаем скоры для всех связей разом
            query_emb = h_proj * r_emb
            all_scores = torch.matmul(query_emb, E_all.T) # [batch_size, N_total]

            #фильтрация
            mask_b = []
            mask_idx = []
            target_scores = torch.zeros(batch_size, device=device)

            #собираме индексы для маски фильтрации
            for i in range(batch_size):
                h_key = (h_loc[i].item(), h_typ[i].item())
                t_key = (t_loc[i].item(), t_typ[i].item())
                r = r_idx[i].item()

                h_eval_id = entity_to_eval_id[h_key]
                target_t_eval_id = entity_to_eval_id[t_key]

                #сохраняем целевой скор
                target_scores[i] = all_scores[i, target_t_eval_id]

                #проверяем наличие свази в графе
                true_tails = filter_dict[(h_eval_id, r)]
                for true_t in true_tails:
                    if true_t != target_t_eval_id:
                        mask_b.append(i)
                        mask_idx.append(true_t)

            #зануляем существующие связи
            if mask_b:
                all_scores[mask_b, mask_idx] = -1e9

            #сравниваем всю матрицу скоров с вектором целевых скоров
            ranks = (all_scores > target_scores.unsqueeze(1)).sum(dim=1) + 1

            #собираем метрики
            mrr += (1.0 / ranks).sum().item()
            for k in k_list:
                hits[k] += (ranks <= k).sum().item()

            del all_scores, query_emb


        mrr = mrr / total_samples
        hits = {k: v/total_samples for k, v in hits.items()}

        return mrr, hits
    

In [ ]:
#читаем предобученные эмбеддинги
with open("../data/dicts/dicts_for_mdm/dict_ESM_650M.pkl", 'rb') as f:
    emb_prot = pickle.load(f)
with open("../data/dicts/dicts_for_mdm/dict_chemberta_77m.pkl", 'rb') as f:
    emb_sm = pickle.load(f)
with open("../data/dicts/dicts_for_mdm/rna_berta.pkl", 'rb') as f:
    emb_rna = pickle.load(f)
with open("../data/dicts/dicts_for_mdm/dna_full.pkl", 'rb') as f:
    emb_dna = pickle.load(f)

#объединяем в один словарь
rawid2enb = emb_prot | emb_sm | emb_rna | emb_dna


node_types = ['AA', 'DNA', 'RNA', 'SmallMolecule']
type2id = {name: i for i, name in enumerate(node_types)}

rawid_to_local = {}
rawid_to_type = {}
raw_tensors = {}


for t_name in node_types:
    #читаем колонку с ID чтобы узнать тип узлов
    df_nodes = pd.read_csv(f'../data/nodes/nodes_for_mdm/{t_name}.csv', usecols=['id_entity'])

    embeddings_list = []

    for local_idx, raw_id in enumerate(df_nodes['id_entity']):
        #заполняем словари для DataLoader
        rawid_to_local[raw_id] = local_idx
        rawid_to_type[raw_id] = type2id[t_name]

        #берем готовый эмбеддинг из исходного словаря
        emb = rawid2enb[raw_id]
        embeddings_list.append(emb)

    #склеиваем список тензоров в одну матрицу для этого типа [N_nodes_of_this_type, embedding_dim]
    raw_tensors[str(type2id[t_name])] = torch.stack(embeddings_list)

df_edges = pd.read_csv('../data/edges/clean_edges_without_NaNm.csv')

#превращаем предикаты в числа
pred_keys, pred_values = pd.factorize(df_edges['predicate'])
df_edges['predicate'] = pred_keys

#мапим сырые ID сразу в локальные индексы и типы
h_local = df_edges['id_entity_1'].map(rawid_to_local).to_numpy()
h_type  = df_edges['id_entity_1'].map(rawid_to_type).to_numpy()
t_local = df_edges['id_entity_2'].map(rawid_to_local).to_numpy()
t_type  = df_edges['id_entity_2'].map(rawid_to_type).to_numpy()
rel_idx = df_edges['predicate'].to_numpy()

#тензоры для DataLoader
h_local_tensor = torch.tensor(h_local, dtype=torch.long)
h_type_tensor  = torch.tensor(h_type, dtype=torch.long)
t_local_tensor = torch.tensor(t_local, dtype=torch.long)
t_type_tensor  = torch.tensor(t_type, dtype=torch.long)
rel_tensor     = torch.tensor(rel_idx, dtype=torch.long)

torch.manual_seed(100)
torch.cuda.manual_seed(100)

#создаем сеты для обучения и оценки
dataset = GraphDataset(h_local_tensor, h_type_tensor, rel_tensor, t_local_tensor, t_type_tensor)
train_set, val_set, test_set = random_split(dataset, [0.8, 0.1, 0.1])
train_loader = DataLoader(dataset=train_set, batch_size=4096, shuffle=True)
test_loader = DataLoader(dataset=test_set, batch_size=128, shuffle=True)



#гиперпараметры
EMB_DIM = 192
LR = 2e-3
MARGIN = 1
WEIGHT = 1e-4
EPOCHS = 5


model = MultiModalDistMult(raw_tensors=raw_tensors, output_dim=EMB_DIM, num_relations=len(pred_values)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay = WEIGHT)
loss_fn = nn.MarginRankingLoss(margin=MARGIN)

model.train()

for epoch in tqdm(range(EPOCHS)):
    for batch in train_loader:
        optimizer.zero_grad()

        #позитивные примеры
        h_loc, h_typ, r_id, t_loc, t_typ = [b.to(device) for b in batch]
        batch_size = h_loc.size(0)

        #считаем скор для настоящих триплетов
        pos_scores = model(h_loc, h_typ, r_id, t_loc, t_typ)

        #генерируем негативные триплеты внутри батча
        perm = torch.randperm(batch_size, device=device)
        neg_t_loc = t_loc[perm]
        neg_t_typ = t_typ[perm]

        #считаем скор для негативным триплетам
        neg_scores = model(h_loc, h_typ, r_id, neg_t_loc, neg_t_typ)

        #считаем ошибку
        target = torch.ones_like(pos_scores)
        loss = loss_fn(pos_scores, neg_scores, target)

        loss.backward()
        optimizer.step()




In [ ]:
#оценка mdm модели
E_all, entity_to_eval_id, filter_dict = prepare_evaluation_mdm(
model, h_local_tensor, h_type_tensor, rel_tensor,
t_local_tensor, t_type_tensor, device)

evaluate_filtered_mdm(model, test_loader, E_all, entity_to_eval_id, filter_dict, device, k_list=[1, 5, 10, 50, 100])

In [7]:
# 1. Находим множитель (максимальный тип + 1)
# Важно: берем максимум и из тензора, и из словаря для безопасности
max_t_tensor = t_type_tensor.max().item()
max_t_dict = max(k[1] for k in entity_to_eval_id.keys())
M = max(max_t_tensor, max_t_dict) + 1

# 2. Переводим ключи словаря в 1D-хеши
dict_keys = []
dict_values = []
for (e_id, e_type), eval_id in entity_to_eval_id.items():
    dict_keys.append(e_id * M + e_type)
    dict_values.append(eval_id)

dict_keys = torch.tensor(dict_keys, dtype=torch.long)
dict_values = torch.tensor(dict_values, dtype=torch.long)

# 3. Создаем 1D lookup тензор
# Размер будет равен максимальному хешу + 1
max_hash = dict_keys.max().item()
lookup_tensor = torch.full((max_hash + 1,), -1, dtype=torch.long)
lookup_tensor[dict_keys] = dict_values

# 4. Векторизованный поиск (МГНОВЕННО)
# Считаем хеши для всех элементов тензора сразу
input_hashes_heads = h_local_tensor * M + h_type_tensor
input_hashes_tales = t_local_tensor * M + t_type_tensor

gnn_heads = lookup_tensor[input_hashes_heads]
gnn_tales = lookup_tensor[input_hashes_tales]

edges = torch.vstack([gnn_heads, gnn_tales])

# Старый код

In [ ]:
#создаем и заполняем объект Data для GNN
gnn_data = Data()
gnn_data.x = E_all
gnn_data.edge_index = edges
gnn_data.edge_attr = rel_tensor

#оцениваем iw связь (можно hs если нужно)
current_link = 'iw'
iw_mask = gnn_data.edge_attr == 0
hs_mask = gnn_data.edge_attr == 1
iw_edge_index = gnn_data.edge_index[:, iw_mask]
hs_edge_index = gnn_data.edge_index[:, hs_mask]

if current_link == 'iw':
    one_link_data = Data(x=gnn_data.x, edge_index=iw_edge_index, edge_arrr=torch.zeros(iw_edge_index.shape[1]))
else:
    one_link_data = Data(x=gnn_data.x, edge_index=hs_edge_index, edge_arrr=torch.zeros(hs_edge_index.shape[1]))


#создаем сплитер данных
transform = T.RandomLinkSplit(
    num_val=0.1, 
    num_test=0.1, 
    is_undirected=True,
    add_negative_train_samples=False,
    neg_sampling_ratio=0,
)

#сплит данных
train_data, val_data, test_data = transform(one_link_data)

#добавляем hs ребра для messadge passing
def add_hs_edges(split_data, add_edge_index):
    split_data.edge_index = torch.cat([split_data.edge_index, add_edge_index], dim=1)
    return split_data

if current_link == 'iw':
    train_data = add_hs_edges(train_data, hs_edge_index)
    val_data = add_hs_edges(val_data, hs_edge_index)
    test_data = add_hs_edges(test_data, hs_edge_index)
else:
    train_data = add_hs_edges(train_data, iw_edge_index)
    val_data = add_hs_edges(val_data, iw_edge_index)
    test_data = add_hs_edges(test_data, iw_edge_index)

In [ ]:
#класс GNN модели
class Gnn_model(nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        #один слой пердобработки
        self.lin0 = nn.LazyLinear(hidden_channels)
        self.act0 = nn.PReLU()

        #два слоя gnn с нормализацией
        self.conv1 = SAGEConv((-1, -1), hidden_channels, aggr = 'sum')
        self.norm1 = nn.LayerNorm(hidden_channels)
        self.act1 = nn.PReLU()
        
        self.conv2 = SAGEConv((-1, -1), hidden_channels, aggr = 'sum')
        self.norm2 = nn.LayerNorm(hidden_channels)
        self.act2 = nn.PReLU()
    
        #два слоя постобработки
        self.lin_out = nn.Sequential(
            nn.LazyLinear(2 * out_channels),
            nn.PReLU(),
            nn.LayerNorm(2 * out_channels),
            nn.LazyLinear(out_channels)
        )

    def encode(self, x, edge_index):
        #предобработка
        h0 = self.lin0(x)
        h0 = self.act0(h0)
        
        #1 слой gnn
        h1 = self.conv1(h0, edge_index)
        h1 = self.norm1(h1)
        h1 = self.act1(h1)
        
        #2 слой gnn
        h2 = self.conv2(h1, edge_index)
        h2 = self.norm2(h2)
        h2 = self.act2(h2)

        #конкатенация всех уровней
        x_final = torch.cat([h0, h1, h2], dim=1)
        x = self.lin_out(x_final)
        
        #нормализация
        return F.normalize(x, p=2, dim=-1)


    #dot декодер
    def decode(self, z, edge_index):
        row, col = edge_index
        value = (z[col] * z[row]).sum(dim=1)
        return value
        
    
    def forward(self, x, edge_index):
        z = self.encode(x, edge_index)
        return self.decode(z, edge_index)


#тренировка
def train_margin(model, train_loader, optimizer, margin=1.5):
    model.train()
    total_loss = 0
    num_pos_total = 0

    for batch in tqdm(train_loader):
        batch = batch.to(device)
        optimizer.zero_grad()

        #получаем эмбеддинги
        z = model.encode(batch.x, batch.edge_index)

        #извлекаем индексы и определяем количество позитивных примеров
        edge_label_index = batch.edge_label_index
        num_pos = (batch.edge_label == 1).sum().item()
        amount = (batch.edge_label == 0).sum().item() // num_pos
        
        #скоры для позитивных ребер
        pos_idx = edge_label_index[:, :num_pos]
        pos_scores = model.decode(z, pos_idx)

        #скоры для негативынх
        neg_idx = edge_label_index[:, num_pos:]
        neg_scores_all = model.decode(z, neg_idx)

        #находим сложные негативы
        neg_scores_reshaped = neg_scores_all.view(num_pos, amount)
        hard_neg_scores, _ = torch.max(neg_scores_reshaped, dim=1)

        target = torch.ones_like(pos_scores)
        loss = F.margin_ranking_loss(pos_scores, hard_neg_scores, target, margin=margin)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * num_pos
        num_pos_total += num_pos

    return total_loss / num_pos_total

In [ ]:
#загрузчик ребер для обучения
train_loader = LinkNeighborLoader(
    data=train_data,
    #берем 25 узлов вокруг целевого, и 10 вокруг каждого из 25
    num_neighbors=[25, 10],
    batch_size=50000,
    edge_label_index=train_data.edge_label_index, 
    edge_label=train_data.edge_label,
    neg_sampling_ratio=10,               
    shuffle=True
)

gnn_model = Gnn_model(hidden_channels=128, out_channels=64).to(device)
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.001)

for i in range(5):
    loss = train_margin(gnn_model, train_loader, optimizer, margin=1)
    print(loss)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [02:28<00:00,  4.50s/it]


0.5837985435522518


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [02:30<00:00,  4.56s/it]


0.42333371885828824


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [02:29<00:00,  4.53s/it]


0.37739530491918266


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [02:31<00:00,  4.59s/it]


0.349754363301761


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [02:29<00:00,  4.53s/it]

0.33230989773503283


In [ ]:
#функция для получения всех эмбеддингов после модели
def get_embeddings_gnn(model, test_data):
    #загрузчик
    node_loader = NeighborLoader(
    test_data,
    num_neighbors=[25, 10],
    batch_size=50000,
    shuffle=False
    )
    
    all_embeddings = []
    
    model.eval()
    with torch.no_grad():
        for batch in tqdm(node_loader):
            batch = batch.to(device)
            z_batch = model.encode(batch.x, batch.edge_index)
        
            all_embeddings.append(z_batch[:batch.batch_size].cpu())
    
    #объединяем все части в одну матрицу
    full_embedding_matrix = torch.cat(all_embeddings, dim=0)
    return full_embedding_matrix

#функция для создания словаря фильтрации
def get_filtred_dict_gnn(data):
    filter_dict = defaultdict(set)

    for i in tqdm(data.edge_index.T):
        filter_dict[i[0].item()].add(i[1].item())

    return filter_dict

#общая функция подготовки
def prepare_evaluation_gnn(model, data, test_data):
    all_embeddings = get_embeddings_gnn(model, test_data)
    filter_dict = get_filtred_dict_gnn(data)

    return all_embeddings, filter_dict

final_EMB, filtr_gnn = prepare_evaluation_gnn(gnn_model, gnn_data, test_data)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5064442/5064442 [01:52<00:00, 44940.74it/s]


In [ ]:

#функция для тестирования GNN модели@torch.no_grad()
def test_gnn(model, test_data, batch_size, all_embeddings, filter_dict, k_list=[1, 5, 10, 50, 100], device='cuda'):
    model.eval()
    mrr = 0  #средний обратный ранг
    hits = {k: 0 for k in k_list}  #счетчики hits@k
    total_samples = 0

    loader = DataLoader(test_data.edge_label_index.T, batch_size=batch_size, shuffle=False)
    for batch in tqdm(loader):
        batch = batch.to(device)
        corr_batch_size = batch.size(0)  #реальный размер батча (для последнего может быть меньше)
        total_samples += corr_batch_size

        #индексы голов и хвостов
        heads_index = batch[:, 0]  
        targets_index = batch[:, 1]

        #все эмбеддинги сущностей
        all_embeddings = all_embeddings.to(device)  
        #эмбеддинги голов для текущего батча
        heads = all_embeddings[heads_index]  
        #скалярные произведения для оценки всех хвостов разом   
        all_scores = torch.matmul(heads, all_embeddings.T)  

        #фильтруем батчами
        mask_b = []  
        mask_idx = []  
        target_scores = torch.zeros(corr_batch_size, device=device)

        for i in range(corr_batch_size):
            h_eval_id = heads_index[i].item()
            target_t_eval_id = targets_index[i].item()

            #сохраняем целевой скор
            target_scores[i] = all_scores[i, target_t_eval_id]

            #создаем маску истинных триплетов
            true_tails = filter_dict[h_eval_id]
            for true_t in true_tails:
                if true_t != target_t_eval_id:
                    mask_b.append(i)
                    mask_idx.append(true_t)

        #зануляем оценки для настоящих хвостов
        if mask_b:
            all_scores[mask_b, mask_idx] = -1e9  
        
         #ранг целевого хвоста
        ranks = (all_scores > target_scores.unsqueeze(1)).sum(dim=1) + 1 

        #собираем итоговый словарь
        mrr += (1.0 / ranks).sum().item() 
        for k in k_list:
            hits[k] += (ranks <= k).sum().item()

        del all_scores 

    mrr = mrr / total_samples
    hits = {k: v / total_samples for k, v in hits.items()} 
    formatted_hits = {k: f"{v:.4f}" for k, v in hits.items()} 
    
    return {"MRR": mrr, "Hits": formatted_hits}


test_gnn(gnn_model, test_data, batch_size = 256, all_embeddings = final_EMB, filter_dict = filtr_gnn)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 784/784 [09:01<00:00,  1.45it/s]


{'MRR': 0.00870446329332773,
 'Hits': {1: '0.0000', 5: '0.0134', 10: '0.0240', 50: '0.0624', 100: '0.0922'}}

Average GraphSAGE (sum, PReLU, dot), hid = 128, out = 64, marg = 1, 
 'Hits': {1: '0.0067', 5: '0.2183', 10: '0.3041', 50: '0.4398', 100: '0.4826'}}

IW GraphSAGE (sum, PReLU, dot), hid = 128, out = 64, marg = 1, 
 'Hits': {1: '0.0000', 5: '0.0136', 10: '0.0236', 50: '0.0626', 100: '0.0913'}}